# Practical work presentation

Data is from a practical work of neurosciences (M. Aarabi) in my third year of licence. This work is meant to appreciate the impact of various emotions stimuli on our way of looking faces. Data was acquired through Tobii eyetracking devices and using myself (GR=37).


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('C:/Users/Aurélien/Desktop/panda/data_GR37.csv')
print('Setup complete !')

In [ ]:
print(df.shape)
df.head(10)

# Data cleaning

I chose to split the main .csv into two dataframe, containing fixation count or fixation  duration data.

In [ ]:
# Deletion of empty rows and ligns

df.drop('Unnamed: 0', axis=1, inplace=True)
df.drop('Unnamed: 1', axis=1, inplace=True)

df.drop(0, inplace=True)
df.drop(1, inplace=True)
df.drop(5, inplace=True)
df.drop(6, inplace=True)
df.drop(7, inplace=True)

# Deletion of 'All recordings' ligns

df.drop(4, inplace=True)
df.drop(10, inplace=True)

# Empty values replacement

df = df.replace(['-', 'NaN', 'null'], '', regex=True)

# Transposition

df_transposed = df.T
df_transposed.head(10)
df_transposed.columns = ['Fixation count ID', 'Count value', 'Fixation duration ID', 'Duration value']
df_transposed = df_transposed.reset_index(drop=True)

In [ ]:
# Seperating count and duration values from the original dataframe

df_count = df_transposed[['Fixation count ID', 'Count value']]
df_duration = df_transposed[['Fixation duration ID', 'Duration value']]

print(df_count)
print(df_duration)

# Data treatment

We're adding additional columns for the different emotions submitted and the region of interest.

In [ ]:
# Adding the additionnal rows to each dataframe

df_count.insert(loc=2, column='Emotion', value=0)
df_count.insert(loc=3, column='ROI', value=0)

df_duration.insert(loc=2, column='Emotion', value=0)
df_duration.insert(loc=3, column='ROI', value=0)

print(df_count)
print(df_duration)

According to M. Aarabi guidelines :

Emotions :
- Neutral: all files ending in …NES.JPG
- Happy: all files ending in …HAS.JPG
- Angry: all files ending in …ANS.JPG
- Sad: all files ending in … SAS.JPG

Regions of interest:
- Right eye: ROI 1, 5, 9, 13, 17, 21, 25, 29, 33, 37, 41, 45, 49, 53, 57, 61, 65, 69, 73, 77, 81, 85, 89, 93, 97, 101, 105, 109, 113, 117, 121, 125
- Left eye: ROI 2, 6, 10, 14, 18, 22, 26, 30, 34, 38, 42, 46, 50, 54, 58, 62, 66, 70, 74, 78, 82, 86, 90, 94, 98, 102, 106, 110, 114, 118, 122, 126
- Mouth: ROI 3, 7, 11, 15, 19, 23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63, 67, 71, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 127
- Nose: ROI 4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 96, 100, 104, 108, 112, 116, 120, 124, 128
- Area between the right eye and the left eye: R1, …, R32

In [ ]:
df_count.columns.get_loc('Fixation count ID')

In [ ]:
# Emotions treatment function

def df_emotion_treatment(df):

    col_id = df.columns[0]

    conditions = [
        df[col_id].str.contains('NES', case=False, na=False),
        df[col_id].str.contains('HAS', case=False, na=False),
        df[col_id].str.contains('ANS', case=False, na=False),
        df[col_id].str.contains('SAS', case=False, na=False)
    ]
    emotions = ['Neutral', 'Happy', 'Angry', 'Sad']
    
    df['Emotion'] = np.select(conditions, emotions, default='Unknown')
    
    return df


In [ ]:
df_count = df_emotion_treatment(df_count)
print(df_count)


In [ ]:
df_duration = df_emotion_treatment(df_duration)
print(df_duration)

In [ ]:
# ROI treatment

right_eye = [1, 5, 9, 13, 17, 21, 25, 29, 33, 37, 41, 45, 49, 53, 57, 61, 65, 69, 73, 77, 81, 85, 89, 93, 97, 101, 105, 109, 113, 117, 121, 125]
left_eye = [2, 6, 10, 14, 18, 22, 26, 30, 34, 38, 42, 46, 50, 54, 58, 62, 66, 70, 74, 78, 82, 86, 90, 94, 98, 102, 106, 110, 114, 118, 122, 126]
mouth = [3, 7, 11, 15, 19, 23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63, 67, 71, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 127]
nose = [4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 96, 100, 104, 108, 112, 116, 120, 124, 128]

def df_ROI_treatment(df):

    col_id = df.columns[0]
    
    num_roi = df[col_id].str.extract(r'_ROI(\d+)_')[0].astype(float)
    
    conditions_roi = [
        num_roi.isin(right_eye),
        num_roi.isin(left_eye),
        num_roi.isin(mouth),
        num_roi.isin(nose),
        df[col_id].str.contains(r'_R\d+_', regex=True)
    ]
    
    noms_roi = ['Right eye', 'Left eye', 'Mouth', 'Nose', 'Area between eyes']
    df['ROI'] = np.select(conditions_roi, noms_roi, default='Unknown')

    return df

In [ ]:
df_count = df_ROI_treatment(df_count)
print(df_count)

In [ ]:
df_duration = df_ROI_treatment(df_duration)
print(df_duration)